# 01 — Build training features from public international results

This notebook builds the football feature table from scratch.

Input:
- downloads `results.csv` from the public international-results dataset.

Output:
- `data/processed/training_features.csv`
- `data/processed/training_features_with_historical_external.csv`
- `data/processed/01_training_features_report_bundle.zip`

The Odds API calls: 0.


In [ ]:
# Cell 1. Imports and helpers.


from pathlib import Path
import json
import re
import zipfile
import time

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42

TEAM_REPLACEMENTS = {
    "USA": "United States",
    "US": "United States",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",
}

def norm_team(name):
    if pd.isna(name):
        return ""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return TEAM_REPLACEMENTS.get(name, name)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            payload,
            f,
            indent=2,
            ensure_ascii=False,
            default=str,
        )
    return path

def infer_outcome(home_goals, away_goals):
    if pd.isna(home_goals) or pd.isna(away_goals):
        return np.nan
    if home_goals > away_goals:
        return 2
    if home_goals < away_goals:
        return 0
    return 1

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10 ** (-(rating_a - rating_b) / 400.0))


print("Setup OK.")


In [ ]:
# Cell 2. Download public international results.

RESULTS_URL = (
    "https://raw.githubusercontent.com/"
    "martj42/international_results/master/results.csv"
)

results_path = RAW_DIR / "international_results.csv"

if results_path.exists():
    results = pd.read_csv(results_path)
    print("Loaded existing:", results_path)
else:
    results = pd.read_csv(RESULTS_URL)
    results.to_csv(results_path, index=False)
    print("Downloaded and saved:", results_path)

print("Rows:", len(results))
display(results.head())


In [ ]:
# Cell 3. Standardize results.

required = {
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "neutral",
}

missing = required - set(results.columns)
if missing:
    raise ValueError(f"Missing columns in results.csv: {missing}")

matches = results.copy()

matches["date"] = pd.to_datetime(matches["date"], errors="coerce")
matches["home_team"] = matches["home_team"].map(norm_team)
matches["away_team"] = matches["away_team"].map(norm_team)
matches["home_score"] = pd.to_numeric(
    matches["home_score"],
    errors="coerce",
)
matches["away_score"] = pd.to_numeric(
    matches["away_score"],
    errors="coerce",
)
matches["neutral"] = matches["neutral"].fillna(True).astype(bool)

matches = matches.dropna(
    subset=[
        "date",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
    ]
).copy()

matches["outcome"] = matches.apply(
    lambda r: infer_outcome(r["home_score"], r["away_score"]),
    axis=1,
)

matches = matches.dropna(subset=["outcome"]).copy()
matches["outcome"] = matches["outcome"].astype(int)

matches = matches.sort_values("date").reset_index(drop=True)

matches.to_csv(
    PROCESSED_DIR / "matches_standardized.csv",
    index=False,
)

print("Standardized matches:", matches.shape)
display(matches.head())


In [ ]:
# Cell 4. Build Elo and rolling features.

K = 30.0
ratings = {}
recent = {}
last_date = {}

rows = []

for _, r in matches.iterrows():
    date = r["date"]
    h = r["home_team"]
    a = r["away_team"]
    hg = float(r["home_score"])
    ag = float(r["away_score"])
    is_neutral = bool(r["neutral"])

    rh = ratings.get(h, 1500.0)
    ra = ratings.get(a, 1500.0)
    exp_h = expected_score(rh, ra)

    def rolling_stats(team):
        hist = recent.get(team, [])[-5:]
        if not hist:
            return {
                "points": 0.0,
                "gf": 0.0,
                "ga": 0.0,
                "n": 0,
            }
        return {
            "points": float(np.mean([x["points"] for x in hist])),
            "gf": float(np.mean([x["gf"] for x in hist])),
            "ga": float(np.mean([x["ga"] for x in hist])),
            "n": len(hist),
        }

    hs = rolling_stats(h)
    aas = rolling_stats(a)

    h_last = last_date.get(h)
    a_last = last_date.get(a)

    rest_h = (date - h_last).days if h_last is not None else np.nan
    rest_a = (date - a_last).days if a_last is not None else np.nan

    rows.append({
        "date": date,
        "home_team": h,
        "away_team": a,
        "home_score": hg,
        "away_score": ag,
        "outcome": int(r["outcome"]),
        "tournament": r["tournament"],
        "neutral": is_neutral,
        "is_neutral": int(is_neutral),
        "is_home_not_neutral": int(not is_neutral),
        "elo_home_pre": rh,
        "elo_away_pre": ra,
        "elo_diff": rh - ra,
        "elo_home_expected": exp_h,
        "home_form_points_per_match_5": hs["points"],
        "away_form_points_per_match_5": aas["points"],
        "diff_form_points_per_match_5": hs["points"] - aas["points"],
        "home_goals_for_avg_5": hs["gf"],
        "away_goals_for_avg_5": aas["gf"],
        "diff_goals_for_avg_5": hs["gf"] - aas["gf"],
        "home_goals_against_avg_5": hs["ga"],
        "away_goals_against_avg_5": aas["ga"],
        "diff_goals_against_avg_5": hs["ga"] - aas["ga"],
        "home_recent_matches": hs["n"],
        "away_recent_matches": aas["n"],
        "home_rest_days": rest_h,
        "away_rest_days": rest_a,
        "diff_rest_days": (
            rest_h - rest_a
            if not pd.isna(rest_h) and not pd.isna(rest_a)
            else np.nan
        ),
    })

    actual_h = 1.0 if hg > ag else 0.0 if hg < ag else 0.5

    ratings[h] = rh + K * (actual_h - exp_h)
    ratings[a] = ra + K * ((1.0 - actual_h) - (1.0 - exp_h))

    hp, ap = (3, 0) if hg > ag else (0, 3) if hg < ag else (1, 1)

    recent.setdefault(h, []).append({"gf": hg, "ga": ag, "points": hp})
    recent.setdefault(a, []).append({"gf": ag, "ga": hg, "points": ap})

    last_date[h] = date
    last_date[a] = date

training_features = pd.DataFrame(rows)

training_features.to_csv(
    PROCESSED_DIR / "training_features.csv",
    index=False,
)

training_features.to_csv(
    PROCESSED_DIR / "training_features_with_historical_external.csv",
    index=False,
)

print("Training features:", training_features.shape)
display(training_features.head())


In [ ]:
# Cell 5. Save report zip.

REPORT_DIR = PROCESSED_DIR / "01_training_features_report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "source_results_path": str(results_path),
    "matches_rows": int(len(matches)),
    "training_features_rows": int(len(training_features)),
    "training_features_columns": list(training_features.columns),
}

save_json(REPORT_DIR / "summary.json", summary)

training_features.head(100).to_csv(
    REPORT_DIR / "training_features_head.csv",
    index=False,
)

zip_path = PROCESSED_DIR / "01_training_features_report_bundle.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in REPORT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(REPORT_DIR))
    zf.write(
        PROCESSED_DIR / "training_features.csv",
        arcname="training_features.csv",
    )

print("Created:", zip_path.resolve())
